## Import files

In [1]:
# Import data processing modules
import pandas as pd # used throughout the code
import numpy as np # used throughout the code
import country_converter as coco # used for converting country names to ISO codes
from sklearn.impute import KNNImputer # used for imputation of values for missing country-climate zone combinations
#import scipy as sp # used for correlation analysis after disaggregation


# Import visualization modules
import matplotlib.pyplot as plt # used for simple bar charts and as basis for plotly and seaborn library
import plotly.express as px # used for all stacked bar charts
import plotly.graph_objects as go # used for marimekko plot and for annotating stacked bar charts
import seaborn as sns # used for scatterplot and kdeplot in data and sensitivity analysis section after disaggregation
from textwrap import fill # used for scatterplot in data analysis section after disaggregation

# Display options
pd.options.display.float_format = '{:,.2f}'.format #limits printed decimal points to two
#pd.reset_option('^display.', silent=True) #option to reset the previous display option

In [2]:
urbanity = False #adds a third urbanity category ("sub": suburban) to the urban-rural typology (urt)
if urbanity == True:
    resolution = '_nuts_sub'
else:
    resolution = '_nuts'

urt_activate = False #enables national level reaggregation of NUTS level data on occupancy using climate zones and broad urbanity category of NUTS region

climate_calc = False #set this to true to reproduce climate zone with NUTS region matching

if climate_calc == True:
    import geopandas as gpd
    import xarray as xr #used to open netcdf climate file

    from matplotlib.colors import ListedColormap
    from matplotlib import colormaps
    from matplotlib.gridspec import GridSpec
    from matplotlib import rcParams

    #%% Global plotting settings
    rcParams['font.family'] = 'Bitstream Vera Sans'
    rcParams['font.size'] = 10

## Add Approach: add another column with NUTS3 labels to regions label file

In [3]:
#Functions

def nuts3code_to_region_nuts(input):
      input = pd.merge(input,code_to_region_nuts, how='left') #adding region labels
      for nuts3code in input[input['region_nuts'].isna()]['NUTS-3 Code']:
            if nuts3code in code_to_region_nuts['code_2021'].values: #adding region labels with name changes from 2021 to 2024
                  input.loc[input['NUTS-3 Code']==nuts3code,'region_nuts'] = code_to_region_nuts.loc[code_to_region_nuts['code_2021'] == nuts3code, 'region_nuts'].values[0]
                  input.loc[input['NUTS-3 Code']==nuts3code,'region_bld'] = code_to_region_nuts.loc[code_to_region_nuts['code_2021'] == nuts3code, 'region_bld'].values[0]
                  #not included are Extra-Regio regions (ZZZ), i.e. air and waterways, as well as Switzerland and Norway
      input.drop(['NUTS-3 Code', 'code_1999', 'code_2003', 'code_2006', 'code_2010', 'code_2013',
       'code_2016', 'code_2021'], axis=1, inplace=True)
      input.dropna(subset='region_nuts', inplace=True)
      return input

def all_rows_contained(df1, df2):
    """Check if all rows in df1 are contained in df2"""
    merged = df1.merge(df2, how='left', indicator=True)
    return (merged['_merge'] == 'both').all()

### Expanding the index

In [4]:
# Region Labels Detailed: creating labels that include an extra level for NUTS3 regions

#importing NUTS labels
if urt_activate == True:
    usecols = ['Country code', 'NUTS-3 Code', 'Urban-Rural typology']
else:
    usecols = ['Country code', 'NUTS-3 Code']
nuts_lab = pd.read_excel('data/input_detailing_NUTS/NUTS2021-NUTS2024.xlsx', sheet_name = 'NUTS-3 Typologies', header=0, usecols=usecols)
nuts_lab['iso3'] = coco.convert(names=nuts_lab['Country code'], to='ISO3')

#importing region labels
region_lab = pd.read_csv('data/input_csv_SSP_2023_resid/regions_R61.csv')
region_lab['iso3']=region_lab.region_bld.str[-3:]

#appending NUTS labels to region labels where available
region_nuts_lab = pd.merge(nuts_lab, region_lab, on='iso3',how='left') #'outer'
region_nuts_lab['region_nuts'] = [str(x) + '-' + str(y) for x, y in zip(region_nuts_lab['region_bld'], region_nuts_lab['NUTS-3 Code'])]
region_nuts_lab['region_nuts'] = region_nuts_lab['region_nuts'].str.strip('-nan')


if urt_activate == True:
    region_nuts_lab = region_nuts_lab.replace({'predominantly urban':'urb', 'intermediate':'urb', 'predominantly rural':'rur'})
    region_nuts_lab = region_nuts_lab.rename({'Urban-Rural typology':'urt'}, axis=1)
    
    #adding rural options to LUX, MLT, CYP
    #region_nuts_lab = region_nuts_lab.append(region_nuts_lab[region_nuts_lab['region_nuts'].str.contains('LUX')].assign(urt='rur'), ignore_index=True)
    #region_nuts_lab = region_nuts_lab.append(region_nuts_lab[region_nuts_lab['region_nuts'].str.contains('MLT')].assign(urt='rur'), ignore_index=True)
    #region_nuts_lab = region_nuts_lab.append(region_nuts_lab[region_nuts_lab['region_nuts'].str.contains('CYP')].assign(urt='rur'), ignore_index=True)
    region_nuts_lab_urb = region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1)

    region_nuts_lab.drop(['urt'], axis=1, inplace=True)
    region_nuts_lab.drop_duplicates(inplace=True)

#define index to use when importing data
code_to_region_nuts = region_nuts_lab[['NUTS-3 Code', 'region_nuts', 'region_bld']] #input to the function nuts3code_to_region_nuts()

#Constructing a correspondence table between NUTS3 classification for 2024 and 2021
nuts2021_2024 = pd.read_excel('data/input_detailing_NUTS/NUTS2021-NUTS2024.xlsx', sheet_name = 'NUTS2021- NUTS2024', header=0)
nuts2021_2024['Code 2021'] = nuts2021_2024['Code 2021'].fillna(method='ffill')
nuts2021_2024['Code 2024'] = nuts2021_2024['Code 2024'].fillna(method='bfill')
nuts2021_2024 = nuts2021_2024.rename(columns={'Code 2021':'code_2021', 'Code 2024':'code_2024'})
nuts2021_2024 = nuts2021_2024[nuts2021_2024['NUTS level']==3]

nuts_changes = pd.read_csv('data/input_detailing_NUTS/nuts_changes.csv')
nuts_changes = nuts_changes[nuts_changes['typology']=='nuts_level_3']

nuts_changes_2024 = pd.merge(nuts_changes, nuts2021_2024, on='code_2021', how='right').sort_index(axis=1).loc[:,'code_1999':'code_2024']
nuts_changes_2024 = nuts_changes_2024.fillna(method='ffill')
nuts_changes_2024 = nuts_changes_2024.rename(columns={'code_2024':'NUTS-3 Code'})
code_to_region_nuts = pd.merge(code_to_region_nuts, nuts_changes_2024, on='NUTS-3 Code', how='left')
code_to_region_nuts.to_csv('data/input_detailing_NUTS/code_to_region_nuts.csv', index=False)

#export
region_nuts_lab = region_nuts_lab.drop(['iso3','NUTS-3 Code', 'Country code'], axis=1)
region_nuts_lab.to_csv('data/input_csv_NUTS_2025_resid/regions_R61'+resolution+'.csv', index=False)
region_nuts_lab.to_csv('data/input_csv_NUTS_2025_resid/regions_R61'+resolution+'.csv', index=False)

#checks
print('Missing values: ',region_nuts_lab.isna().sum().sum())
print('Duplicates: ',region_nuts_lab.duplicated().sum().sum())
print('NUTS3 regions: ',len(region_nuts_lab.region_nuts.unique()))
print('Index still contained: ', region_lab.drop('iso3', axis=1)[region_lab.region_bld.isin(region_nuts_lab.region_bld.unique())].sort_values('region_bld').reset_index(drop=True)\
      .equals(region_nuts_lab.drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

Missing values:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_30286/1772547260.py:39: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  nuts2021_2024['Code 2021'] = nuts2021_2024['Code 2021'].fillna(method='ffill')
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_30286/1772547260.py:40: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  nuts2021_2024['Code 2024'] = nuts2021_2024['Code 2024'].fillna(method='bfill')
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_30286/1772547260.py:48: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  nuts_changes_2024 = nuts_changes_2024.fillna(method='ffill')


In [5]:
#Climate Zones: overlaying NUTS regions with matching climate zones

if climate_calc == True:
       #import climate zones file
       ds = xr.open_dataset('data/input_detailing_NUTS/climate_zones.nc')
       df = ds.to_dataframe()

       #assigning literal climate zone names to match with the clim variable
       df = df.combined
       climate_zones_names = pd.read_csv('data/input_detailing_NUTS/climate_zones_names.csv', header=None)
       climate_zones_names.columns = ['combined', 'clim']
       climate_zones_names = climate_zones_names.iloc[1:]
       replacement_map = pd.Series(climate_zones_names.clim.values, index=climate_zones_names.combined).to_dict()
       df = df.replace(replacement_map)
       df.to_csv('data/input_detailing_NUTS/climate_zones.csv')

       # merge climate zones with NUTS regions, assigning each NUTS region the climate zone that is most common to the area, in case of equal split, one of the zones is selected randomly
       climate = pd.read_csv('data/input_detailing_NUTS/climate_zones.csv')
       gdf = gpd.GeoDataFrame(climate,geometry=gpd.points_from_xy(climate.lon,climate.lat)).drop(['lat', 'lon'], axis=1)
       nuts = gpd.read_file('data/input_detailing_NUTS/NUTS_RG_20M_2024_3035.gpkg') #need to match the coordinate systems between gdf and nuts
       nuts = nuts.loc[nuts['NUTS_ID'].str.len() == 5] #only NUTS3 level
       gdf = gdf.set_crs(epsg=4326)
       gdf = gdf.to_crs(epsg=3035)
       nuts_climate = nuts.sjoin_nearest(gdf, how='left')
       nuts_climate = nuts_climate.drop(['index_right', 'geometry', 'LEVL_CODE', 'CNTR_CODE', 'NAME_LATN', 'NUTS_NAME',
              'MOUNT_TYPE', 'URBN_TYPE', 'COAST_TYPE'], axis=1).groupby(by=['NUTS_ID']).agg(lambda x: pd.Series.mode(x)[0]).reset_index()
       nuts_climate.columns = ['NUTS-3 Code', 'clim']
       nuts_climate.to_csv('data/input_detailing_NUTS/climate_nuts.csv', index=False)

input = pd.read_csv('data/input_detailing_NUTS/climate_nuts.csv', index_col=[0])

input = nuts3code_to_region_nuts(input)

#import file
filename= 'climatic_zones_rev'
input_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')
if urbanity == True:
       input_original = pd.concat([input_original,input_original[input_original['urt']=='urb'].replace('urb', 'sub')]).sort_values('region_bld')

#harmonize index
input_nuts = pd.merge(input_original.drop('clim', axis=1).drop_duplicates(), input, on=['region_bld'], how='right')

#export
if input_nuts['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT']).sum()>0:
       input_nuts = input_nuts[~input_nuts['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
elif input_nuts.duplicated().any():
       input_nuts.drop_duplicates(inplace=True)
       print('dropped duplicates')
else:
       print('no double entries')
input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv', index=False)

#index
region_nuts_clim = input_nuts.copy()

#checks
print('Climate zones missing from EU-NUTS dataset:', set(input_original.clim.unique()) - set(input_nuts.clim.unique()))
print('Regions in which missing climate zones occur:', input_original[input_original['clim'].isin(set(input_original.clim.unique()) - set(input_nuts.clim.unique()))].region_bld.unique())

#checks
print('Missing values: ',input_nuts.isna().sum().sum())
print('Duplicates: ',input_nuts.duplicated().sum().sum())
print('NUTS3 regions: ',len(input_nuts.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(input_original[input_original.region_bld.isin(region_nuts_lab.region_bld.unique())].sort_values('region_bld').reset_index(drop=True), input_nuts.drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

dropped duplicates
Climate zones missing from EU-NUTS dataset: {'Zone_4B Mixed Dry', 'Zone_5B Cool Dry', 'Zone_1B Very hot Dry', 'Zone_7B Very cold Dry', 'Zone_6B Cold Dry', 'Zone_0B Extremely hot Dry'}
Regions in which missing climate zones occur: ['R32BRA' 'R32CAS-CAU' 'R32CAS-OTH' 'R32CHN' 'R32IND' 'R32MEA-H'
 'R32MEA-M' 'R32MEX' 'R32NAF' 'R32OAS-L-PAS' 'R32PAK' 'R32SSA-L'
 'R32SSA-M']
Missing values:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [39]:
    #Check which country-climatezone combinations only appear when having NUTS detail --> important to always use detailed clim file
    mapped = input_nuts.copy()

    # OLD CODE ONLY FOR COMPARISON! #TODO: remove this

    #import file
    filename= 'climatic_zones_rev'
    input = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')

    #merge files
    input_nuts = pd.merge(input, region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1), on=['region_bld'], how='right')

    duplic = input_nuts.copy()

    # Finding combinations that only appear in one of two dataframes

    # Get unique combinations for each dataframe
    duplic_combinations = set(zip(duplic['region_bld'], duplic['clim']))
    mapped_combinations = set(zip(mapped['region_bld'], mapped['clim']))

    # Find combinations that are only in duplic (not in mapped)
    only_in_duplic = duplic_combinations - mapped_combinations

    # Find combinations that are only in mapped (not in duplic)
    only_in_mapped = mapped_combinations - duplic_combinations

    # All combinations that don't appear in both
    not_in_both = only_in_duplic.union(only_in_mapped)

    print("Combinations only in duplic:")
    for combo in only_in_duplic:
        print(f"  region_bld: {combo[0]}, clim: {combo[1]}")

    print("\nCombinations only in mapped: "+str(len(only_in_mapped)))
    for combo in only_in_mapped:
        print(f"  region_bld: {combo[0]}, clim: {combo[1]}")

Combinations only in duplic:

Combinations only in mapped: 26
  region_bld: C-WEU-BEL, clim: Zone_5A Cool Humid
  region_bld: C-WEU-DEU, clim: Zone_6A Cold Humid
  region_bld: C-WEU-FRA, clim: Zone_1A Very hot Humid
  region_bld: C-WEU-GRC, clim: Zone_5A Cool Humid
  region_bld: C-WEU-FRA, clim: Zone_2A Hot Humid
  region_bld: C-EEU-SVN, clim: Zone_4A Mixed Humid
  region_bld: C-WEU-IRL, clim: Zone_4A Mixed Humid
  region_bld: C-WEU-GRC, clim: Zone_4C Mixed Marine
  region_bld: C-WEU-PRT, clim: Zone_4A Mixed Humid
  region_bld: C-WEU-SWE, clim: Zone_8A Subarctic/arctic Humid
  region_bld: C-WEU-AUT, clim: Zone_7A Very cold Humid
  region_bld: C-EEU-HRV, clim: Zone_4C Mixed Marine
  region_bld: C-WEU-FRA, clim: Zone_3C Warm Marine
  region_bld: C-WEU-GRC, clim: Zone_5C Cool Marine
  region_bld: C-WEU-FRA, clim: Zone_0A Extremely hot Humid
  region_bld: C-WEU-ITA, clim: Zone_7A Very cold Humid
  region_bld: C-WEU-SWE, clim: Zone_7A Very cold Humid
  region_bld: C-EEU-SVN, clim: Zone_7A V

In [29]:
#Check which files contain clim as index --> most files do
import os
import glob

folder_path = "/Users/Mira/Desktop/MCC/Code/EUBUCCO-IAM/message-ix-buildings/message_ix_buildings/sturm/data/input_csv_NUTS_2025_resid"#"/Users/Mira/Desktop/MCC/Code/EUBUCCO-IAM/messageix-buildings-subnational/2025_EU/input_resid"
files_with_clim = [os.path.basename(f) for f in glob.glob(os.path.join(folder_path, "*.csv")) if 'clim' in pd.read_csv(f, nrows=0).columns]
print("Files with 'clim' column:")
for filename in files_with_clim:
    print(f"  {filename}")

Files with 'clim' column:
  stock_baseyear_resid_rev_share_nuts_bld.csv
  bld_shr_access_cool_resid_ssp2_rev.csv
  bld_shr_access_cool_resid_ssp2_rev_nuts.csv
  bld_share_mat_resid_ssp2_rev.csv
  shr_need_heat_resid_rev_nuts_bld.csv
  pop_clim_rev_SSP2_nuts_bld.csv
  shr_need_cool_resid_rev_nuts_bld.csv
  bld_share_mat_resid_ssp2_rev_nuts_bld.csv
  shr_need_heat_resid_rev.csv
  heat_intensity_rev_nuts_bld.csv
  stock_baseyear_resid_rev_share_nuts.csv
  climatic_zones_rev_nuts.csv
  climatic_zones_rev_nuts_bld.csv
  bld_shr_access_cool_resid_ssp2_rev_nuts_bld.csv
  heat_intensity_rev_nuts.csv
  cool_days_nuts_bld.csv
  cool_intensity_rev_nuts.csv
  bld_share_mat_resid_ssp2_rev_nuts.csv
  shr_need_heat_resid_rev_nuts.csv
  shr_need_cool_resid_rev_nuts.csv
  heat_intensity_rev.csv
  pop_clim_rev_SSP2.csv
  cool_days_nuts.csv
  cool_days.csv
  climatic_zones_rev.csv
  shr_need_cool_resid_rev.csv
  stock_baseyear_resid_rev_nuts.csv
  cool_days_rev_nuts.csv
  pop_clim_rev_SSP2_nuts.csv
  hea

### Adding NUTS region specific data: stock composition

In [6]:
# STOCK BASEYEAR SHARES

#Dwelling stock: urbanity, arch, climate zone, income class, year of construction, 

#bld_shr_arch_resid

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWOB_R3
input = pd.read_csv('data/input_detailing_NUTS/estat_cens_21dwob_r3_en.csv', usecols=['Housing', 'Type of building','geo', 'OBS_VALUE']) #dwellings by region and type of building
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['Housing'].isin(['Conventional dwellings', 'Unknown'])] #only occupied or non-occupied, unknown are 0
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch', 'Housing']).sum()
input = input.unstack().droplevel(0, axis=1)
input.columns.name = None
input = input.reset_index().rename({'geo':'NUTS-3 Code'}, axis=1)

input = nuts3code_to_region_nuts(input)
input = pd.merge(input, region_nuts_clim.drop('urt', axis=1).drop_duplicates()) #adding climate zone categories
input = input.groupby(by=['region_bld','region_nuts', 'clim', 'arch']).sum()

dwellings = input.copy()
input = input.div(input.groupby(level=[0,1,2]).sum()) #arch shares of dwellings
input = input.unstack(3).fillna(0.5).stack(future_stack=True) #add missing arch types, and assume equal split

#add average data for missing regions
def add_missing_nuts(df, region_bld, new_nuts):
    climate = region_nuts_clim[region_nuts_clim['region_nuts']==i].clim.unique()[0]
    region_avg = (
        df.groupby(['region_bld', 'clim', 'arch'])
          .mean()
          .loc[(region_bld, climate, slice(None))]
          .assign(region_bld=region_bld, region_nuts=new_nuts, clim=climate)
          .set_index(['region_bld', 'region_nuts', 'clim'], append=True)
          .reorder_levels(['region_bld', 'region_nuts', 'clim', 'arch'])
    )
    return pd.concat([df, region_avg]).sort_index()

for i in set(region_nuts_lab.region_nuts) - set(input.reset_index().region_nuts):
    input = add_missing_nuts(input, i[:9], i)

#dwelling stock national distribution
filename = 'bld_shr_arch_resid'
input_trend_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')
input_trend_original = input_trend_original[input_trend_original.region_gea.isin(region_nuts_lab.region_gea.unique())]
if urbanity == True:
       input_trend_original = pd.concat([input_trend_original,input_trend_original[input_trend_original['urt']=='urb'].replace('urb', 'sub')]).sort_values('region_gea')

#adding years
input_trend_out = pd.concat({year: input for year in input_trend_original.year.unique()},names=['year'])
#adding urban and rural typology
input_trend_out = pd.concat({urt: input_trend_out for urt in input_trend_original.urt.unique()},names=['urt']).reset_index()
#add mat
input_trend_out['mat'] = 'perm'

#align column order with input trend original, and fill missing values (only in 1 NUTSxarch combination)
input_trend_out.fillna(0).to_csv('data/input_csv_NUTS_2025_resid/bld_shr_arch_unoccupied'+resolution+'.csv', index=False)
input_trend_out = input_trend_out.fillna(0).rename({'Occupied conventional dwellings':'value'}, axis=1).drop('Unoccupied conventional dwellings', axis=1)

#ensuring shares add up to 1
input_trend_out = input_trend_out.set_index(input_trend_out.columns.drop('value').to_list()).unstack('arch')
input_trend_out[('value', 'sfh')] = 1-input_trend_out[('value', 'mfh')]
input_trend_out = input_trend_out.stack('arch').reset_index()
input_trend_out = input_trend_out.reindex(columns=['region_bld', 'region_nuts', 'urt', 'mat', 'arch', 'year', 'value'])

#export csv
input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv', index=False)

#checks
original = input_trend_original.copy()
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop(['value', 'region_gea'], axis=1).reset_index(drop=True),\
                                                                    out.drop(['value', 'region_bld'], axis=1).drop('region_nuts', axis=1).drop_duplicates().reset_index(drop=True)))

if urbanity == True:
    #only for national recalibration, not for NUTS regions since dwelling stock by urbanity and NUTS region not provided
    pop_by_arch_degurba = pd.read_csv('data/input_detailing_NUTS/ilc_lvho01_page_linear.csv', usecols=['building', 'deg_urb','geo', 'OBS_VALUE'], index_col=[0,1,2])
    hhsize_by_degurba = pd.read_csv('data/input_detailing_NUTS/hbs_car_t315_page_linear.csv', usecols=['deg_urb','geo', 'OBS_VALUE'], index_col=[0,1])
    dw_by_arch_degurba = pop_by_arch_degurba.div(hhsize_by_degurba)
    dw_by_arch_degurba.update(pop_by_arch_degurba, overwrite=False)
    dw_by_arch_degurba = dw_by_arch_degurba.drop(['Others', 'Total', np.nan], axis=0, level=0).drop('Total', axis=0, level=1).unstack('building')
    dw_by_arch_degurba = dw_by_arch_degurba.div(dw_by_arch_degurba.sum(1), axis=0).stack(1).reset_index()
    dw_by_arch_degurba['iso3'] = coco.convert(names=dw_by_arch_degurba['geo'], to='ISO3')
    dw_by_arch_degurba = pd.merge(dw_by_arch_degurba, region_lab[['region_bld', 'iso3']], on='iso3',how='left').dropna(subset='region_bld').replace({'Cities':'urb', 'Towns and suburbs':'sub', 'Rural areas':'rur', 'Flat':'mfh', 'House':'sfh'})
    dw_by_arch_degurba = dw_by_arch_degurba.rename({'deg_urb':'urt', 'building':'arch', 'OBS_VALUE':'value'}, axis=1)
    dw_by_arch_degurba = dw_by_arch_degurba[['region_bld', 'urt', 'arch', 'value']]
    dw_by_arch_degurba = dw_by_arch_degurba.set_index(dw_by_arch_degurba.columns.drop('value').to_list())

    #adding years
    input_trend_out = pd.concat({year: dw_by_arch_degurba for year in input_trend_original.year.unique()},names=['year'])
    input_trend_out = input_trend_out.reset_index()
    #add mat
    input_trend_out['mat'] = 'perm'

    #ensuring shares add up to 1
    input_trend_out = input_trend_out.set_index(input_trend_out.columns.drop('value').to_list()).unstack('arch')
    input_trend_out[('value', 'sfh')] = 1-input_trend_out[('value', 'mfh')]
    input_trend_out = input_trend_out.stack('arch').reset_index()
    input_trend_out = input_trend_out.reindex(columns=['region_bld', 'urt', 'mat', 'arch', 'year', 'value'])

    #export csv
    input_trend_out.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_sub.csv', index=False)

    #checks
    original = input_trend_original.copy()
    out = input_trend_out.copy()
    print('Missing values: ',out.isna().sum().sum())
    print('Null before: ',len(original[original.value==0]))
    print('Null after: ',len(out[out.value==0]))
    print('Duplicates: ',out.duplicated().sum().sum())

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_30286/895049128.py:64: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  input_trend_out = input_trend_out.stack('arch').reset_index()


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  False


In [7]:
#construction period
#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWOP_R3
input = pd.read_csv('data/input_detailing_NUTS/estat_cens_21dwop_r3_en.csv', usecols=['Housing', 'y_const','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['Housing'].isin(['Conventional dwellings', 'Unknown'])] #only occupied or non-occupied, unknown are 0
input = input.loc[~input['y_const'].isin(['UNK', 'TOTAL'])] #only direct numbers
input['y_const']=input.y_const.str[-4:]
input.replace({'2016':'2020', '1919':'1945'}, inplace=True)
input = input.groupby(by=['geo', 'y_const', 'Housing']).sum()
input = input.unstack().droplevel(0, axis=1)
input.columns.name = None
input = input.reset_index().rename({'geo':'NUTS-3 Code', 'y_const':'yr_con'}, axis=1)

#adding missing region labels
input = nuts3code_to_region_nuts(input)
input = input.groupby(by=['region_bld','region_nuts', 'yr_con']).sum().reset_index() #aggregating over merged NUTS3 regions
input = input.set_index(['region_bld','region_nuts', 'yr_con']).div(input.groupby(by=['region_bld','region_nuts']).sum().drop('yr_con', axis=1)).reset_index() #TODO: - interpolate input_shares to more detailed years

input = pd.merge(input, region_nuts_clim.drop('urt', axis=1).drop_duplicates()) #adding climate zone categories

input.set_index(['region_bld','region_nuts', 'clim', 'yr_con'], inplace=True)

#add average data for missing regions
def add_missing_nuts(df, region_bld, new_nuts):
    climate = region_nuts_clim[region_nuts_clim['region_nuts']==i].clim.unique()[0]
    if len(df.reset_index()[(df.reset_index()['region_bld']==region_bld) & (df.reset_index()['clim']==climate)])>0:
        region_avg = (
            df.groupby(['region_bld', 'clim', 'yr_con'])
            .mean(numeric_only=True)
            .loc[(region_bld, climate, slice(None))]
            .assign(region_bld=region_bld, region_nuts=new_nuts, clim=climate)
            .set_index(['region_bld', 'region_nuts', 'clim'], append=True)
            .reorder_levels(['region_bld', 'region_nuts', 'clim', 'yr_con'])
        )
    else:
        region_avg = (
            df.groupby(['region_bld', 'yr_con'])
            .mean(numeric_only=True)
            .loc[(region_bld, slice(None))]
            .assign(region_bld=region_bld, region_nuts=new_nuts, clim=climate)
            .set_index(['region_bld', 'region_nuts', 'clim'], append=True)
            .reorder_levels(['region_bld', 'region_nuts', 'clim', 'yr_con'])
        )        
    return pd.concat([df, region_avg]).sort_index()

for i in set(region_nuts_lab.region_nuts) - set(input.reset_index().region_nuts): #{'C-WEU-FRA-FRY50', 'C-WEU-PRT-PT1B0'}
    input = add_missing_nuts(input, i[:9], i)


#dwelling stock national distribution
filename = 'stock_baseyear_resid_rev_share'
input_trend_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')
input_trend_original = input_trend_original[input_trend_original.region_bld.isin(region_nuts_lab.region_bld.unique())]
if urbanity == True:
       input_trend_original = pd.concat([input_trend_original,input_trend_original[input_trend_original['urt']=='urb'].replace('urb', 'sub')]).sort_values('region_bld')

#adding original columns
#adding arch
input_trend_out = pd.concat({arch: input for arch in input_trend_original.arch.unique()},names=['arch'])
#adding urban and rural typology
input_trend_out = pd.concat({urt: input_trend_out for urt in input_trend_original.urt.unique()},names=['urt']).reset_index()
#adding bld age categories
replacement_map = dict(zip(input_trend_original[['yr_con', 'bld_age']].drop_duplicates().yr_con.astype(str), input_trend_original[['yr_con', 'bld_age']].drop_duplicates().bld_age))
input_trend_out['bld_age'] = input_trend_out.yr_con.map(replacement_map)
#add mat
input_trend_out['mat'] = 'perm'
#add gea
input_trend_out['region_gea'] = input_trend_out.region_bld.str[2:5]
#aligning yr_con
input_trend_out['yr_con'] = input_trend_out['yr_con'].astype(int)



#refining annual detail back to input_trend_original

replacement_map = {1945:1945, 1960:1960, 1965:1980, 1970:1980, 1975:1980, 1980:1980, 1985:2000, 1990:2000, 1995:2000, 2000:2000, 2005:2010, 2010:2010, 2015:2015, 2020:2020}
input_trend_original['yr_input'] = input_trend_original.yr_con.map(replacement_map)

#weights for shares of yr_con in yr_input periods
weights = input_trend_original.groupby(input_trend_original.columns.drop(['value']).to_list()).sum().div(input_trend_original.groupby(input_trend_original.columns.drop(['value', 'yr_con', 'bld_age']).to_list()).sum().drop(['yr_con', 'bld_age'], axis=1))

#filling missing shares in weights
df = weights.unstack(['bld_age', 'yr_input', 'yr_con']).fillna({('value', 'p1', 1945, 1945):1,('value', 'p2', 1960, 1960):1, ('value', 'p3', 2015, 2015):1, ('value', 'p3', 2020, 2020):1})

# Step 1: Calculate sum by yr_input (period) for each row
period_sums = df.groupby(level='yr_input', axis=1).sum()

# Step 2: Identify which yr_input already sum to 1 (with tolerance for floating point)
tolerance = 1e-8
periods_complete = period_sums.apply(lambda x: abs(x - 1) < tolerance, axis=0)

# Step 3: Fill missing values with 0 where yr_input sum is already 1
df_filled = df.copy()
for yr_input in df.columns.get_level_values('yr_input').unique():
    for idx in df.index:
        if periods_complete.loc[idx, yr_input]:
            # Fill NaN with 0 for this yr_input in this row
            df_filled.loc[idx, (slice(None), slice(None), yr_input, slice(None))] = \
                df_filled.loc[idx, (slice(None), slice(None), yr_input, slice(None))].fillna(0)

# Step 4: Fill rural rows using urban data where entire yr_input periods are missing
for idx in df_filled.index:
    # Check if this is a rural row
    if idx[2] == 'rur':  # urt is at position 2 (0-indexed)
        # Create corresponding urban index by replacing 'rural' with 'urban'
        urban_idx = (idx[0], idx[1], 'urb', idx[3], idx[4], idx[5])
        
        # Check if urban counterpart exists
        if urban_idx in df_filled.index:
            for yr_input in df_filled.columns.get_level_values('yr_input').unique():
                period_data = df_filled.loc[idx, (slice(None), slice(None), yr_input, slice(None))]
                
                # If entire yr_input period is missing for this rural row
                if period_data.isna().all():
                    # Fill with corresponding urban data
                    urban_data = df_filled.loc[urban_idx, (slice(None), slice(None), yr_input, slice(None))]
                    df_filled.loc[idx, (slice(None), slice(None), yr_input, slice(None))] = urban_data.values

weights = df_filled.stack(['yr_con', 'bld_age', 'yr_input'])

#add average data for missing clim-bld combinations in weights
def add_missing_clim(df, region_bld, climate):
    region_avg = (df.groupby(['region_bld', 'region_gea', 'urt', 'mat', 'arch', 'yr_con', 'bld_age', 'yr_input'])
                .mean(numeric_only=True)
                .loc[(region_bld, slice(None))]
                .assign(region_bld=region_bld, clim=climate)
                .set_index(['region_bld', 'clim'], append=True)
                .reorder_levels(['region_bld', 'region_gea', 'urt', 'clim', 'mat', 'arch', 'yr_con', 'bld_age', 'yr_input']))   
    return pd.concat([df, region_avg]).sort_index()

for i in set(zip(region_nuts_clim.drop(['urt', 'region_nuts'], axis=1).drop_duplicates()['region_bld'], region_nuts_clim.drop(['urt', 'region_nuts'], axis=1).drop_duplicates()['clim'])) - set(zip(weights.reset_index()[['region_bld', 'clim']].drop_duplicates()['region_bld'], weights.reset_index()[['region_bld', 'clim']].drop_duplicates()['clim'])):
    weights = add_missing_clim(weights, i[0], i[1])

#add nuts region detail to weights
weights = region_nuts_clim.drop('urt', axis=1).drop_duplicates().merge(weights.reset_index(), on=['region_bld', 'clim'],how='left')

#aligning input trend out with detailed year structure
df = input_trend_out.rename(columns={'yr_con':'yr_input'})
df = weights[['yr_con', 'yr_input', 'bld_age']].drop_duplicates().merge(df.drop('bld_age', axis=1), on=['yr_input'], how='left')

df = df.set_index(df.columns.drop(['Occupied conventional dwellings','Unoccupied conventional dwellings']).to_list())
weights = weights.set_index(weights.columns.drop('value').to_list())
input_trend_out = df.reorder_levels([i for i in weights.index.names], axis=0).mul(weights.value, axis=0).reset_index().drop('yr_input', axis=1)

#export in original format
df = input_trend_out.copy()
df.to_csv('data/input_csv_NUTS_2025_resid/stock_baseyear_resid_rev_share_unoccupied'+resolution+'.csv', index=False)
input_trend_out = input_trend_out.rename({'Occupied conventional dwellings':'value'}, axis=1).drop('Unoccupied conventional dwellings', axis=1)
#ensuring shares add up to 1
input_trend_out = input_trend_out.set_index(input_trend_out.columns.drop('value').to_list()).unstack(['yr_con', 'bld_age'])
input_trend_out.loc[:,('value', 1945, 'p1')] = input_trend_out[('value', 1945, 'p1')].sub(input_trend_out.sum(1).sub(1))
input_trend_out = input_trend_out.stack(['yr_con', 'bld_age']).reset_index()

input_trend_out = input_trend_out.reindex(columns=['region_bld', 'region_gea', 'region_nuts', 'urt', 'clim', 'mat', 'arch', 'yr_con', 'bld_age', 'value'])
input_trend_out.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv', index=False)

#checks
original = input_trend_original.copy()
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1)[original.region_bld.isin(region_nuts_lab.region_bld.unique())].sort_values('region_bld').reset_index(drop=True), \
                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_30286/1165480227.py:86: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  period_sums = df.groupby(level='yr_input', axis=1).sum()
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_30286/1165480227.py:119: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  weights = df_filled.stack(['yr_con', 'bld_age', 'yr_input'])
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_30286/1165480227.py:152: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  input

Missing values:  0
Null before:  0
Null after:  2202
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [ ]:
# VACANT and STOCK BASE YEAR SHARES for national x climate x urt (aggregated NUTS version)
if urt_activate == True:
    #distribution of dwellings by occupancy
    df = pd.read_csv('data/input_csv_NUTS_2025_resid/bld_shr_arch_unoccupied'+resolution+'.csv')

    df = df.drop('urt', axis=1).merge(region_nuts_lab_urb[['urt', 'region_nuts']], on='region_nuts').drop_duplicates()
    df = df.set_index(df.columns.drop(['Occupied conventional dwellings','Unoccupied conventional dwellings']).to_list()).mul(dwellings.groupby(level=[0,1,2], axis=0).sum()).groupby(level=[0,1,3,4,5,6], axis=0).sum()
    df.to_csv('data/input_csv_NUTS_2025_resid/stock_dwellings_occupancy_nuts.csv')

    #total stock of unoccupied dwellings in base year based on Eurostat data
    stock = pd.read_csv('data/input_csv_NUTS_2025_resid/stock_dwellings_occupancy_nuts.csv')
    stock = stock[stock.year==2020].drop(['Occupied conventional dwellings', 'mat'], axis=1).rename(columns={'Unoccupied conventional dwellings':'value'})
    stock['value'] = stock['value'].round(0)
    stock['region_gea'] = stock['region_bld'].str[2:5]
    stock.to_csv('data/input_csv_NUTS_2025_resid/stock_vacant_base_resid_Eurostat.csv')

    #share of unoccupied dwellings in total dwellings
    rate = df.div(df.sum(1), axis=0)
    rate['Occupied conventional dwellings'] = rate['Occupied conventional dwellings'].fillna(rate.mean()['Occupied conventional dwellings'])
    rate['Unoccupied conventional dwellings'] = rate['Unoccupied conventional dwellings'].fillna(rate.mean()['Unoccupied conventional dwellings'])
    rate['value'] = rate['Unoccupied conventional dwellings'].round(3)
    rate.droplevel('mat', axis=0).drop(['Occupied conventional dwellings','Unoccupied conventional dwellings'], axis=1).reset_index().to_csv('data/input_csv_NUTS_2025_resid/rate_vacant_occ_ssp2.csv', index=False)

    #total stock of unoccupied dwellings in base year based on MESSAGEix-Buildings original data
    rate = pd.read_csv('data/input_csv_NUTS_2025_resid/rate_vacant_occ_ssp2.csv', index_col=[0,1,2,3,4]).xs(2020, axis=0, level=0)
    stock = pd.read_csv('data/input_csv_NUTS_2025_resid/stock_baseyear_resid_rev.csv', index_col=[0,1,2,3,4,5,6,7,8,9])
    vacant_stock = stock.groupby(axis=0, level=['region_bld', 'region_gea', 'urt', 'clim', 'arch', 'year']).sum(numeric_only=True).mul(rate).div(1-rate)
    vacant_stock.update(stock.groupby(axis=0, level=['region_bld', 'region_gea', 'urt', 'clim', 'arch', 'year']).sum(numeric_only=True).mul(1/3), overwrite=False)
    vacant_stock = vacant_stock.dropna()
    vacant_stock.to_csv('data/input_csv_NUTS_2025_resid/stock_vacant_base_resid_original.csv')

    #archetype distribution of unoccupied dwellings
    arch = df.stack().unstack('arch')
    arch = arch.div(arch.sum(1), axis=0)
    arch = arch.stack().unstack(-2)
    arch['Occupied conventional dwellings'] = arch['Occupied conventional dwellings'].fillna(arch.mean()['Occupied conventional dwellings'])
    arch['Unoccupied conventional dwellings'] = arch['Unoccupied conventional dwellings'].fillna(arch.mean()['Unoccupied conventional dwellings'])
    arch['value'] = arch['Unoccupied conventional dwellings'].round(3)
    arch = arch.drop(['Occupied conventional dwellings','Unoccupied conventional dwellings'], axis=1).unstack('arch')
    arch[('value', 'sfh')] = 1-arch[('value', 'mfh')]
    arch = arch.stack('arch').reset_index()
    arch = arch.reindex(columns=['region_bld', 'clim','urt', 'mat', 'arch', 'year', 'value'])
    arch.to_csv('data/input_csv_NUTS_2025_resid/shr_vacant_base_arch_resid.csv', index=False)

    #archetype distribution of occupied dwellings
    arch = df.stack().unstack('arch')
    arch = arch.div(arch.sum(1), axis=0)
    arch = arch.stack().unstack(-2)
    arch['Occupied conventional dwellings'] = arch['Occupied conventional dwellings'].fillna(arch.mean()['Occupied conventional dwellings'])
    arch['Unoccupied conventional dwellings'] = arch['Unoccupied conventional dwellings'].fillna(arch.mean()['Unoccupied conventional dwellings'])
    arch['value'] = arch['Occupied conventional dwellings'].round(3)
    arch = arch.drop(['Occupied conventional dwellings','Unoccupied conventional dwellings'], axis=1).unstack('arch')
    arch[('value', 'sfh')] = 1-arch[('value', 'mfh')]
    arch = arch.stack('arch').reset_index()
    arch['value'] = arch['value'].round(3)
    arch = arch.reindex(columns=['region_bld', 'clim','urt', 'mat', 'arch', 'year', 'value'])
    arch.to_csv('data/input_csv_NUTS_2025_resid/bld_shr_arch_resid_Eurostat.csv', index=False)

    #construction period distribution of dwellings by occupancy
    df = pd.read_csv('data/input_csv_NUTS_2025_resid/stock_baseyear_resid_rev_share_unoccupied'+resolution+'.csv')

    df = df.drop('urt', axis=1).merge(region_nuts_lab_urb[['urt', 'region_nuts']], on='region_nuts').drop_duplicates()
    df = df.set_index(df.columns.drop(['Occupied conventional dwellings','Unoccupied conventional dwellings']).to_list()).mul(dwellings.groupby(level=[0,1,2], axis=0).sum()).groupby(level=['region_bld', 'clim', 'region_gea', 'urt', 'mat', 'arch', 'yr_con', 'bld_age'], axis=0).sum()
    df.to_csv('data/input_csv_NUTS_2025_resid/stock_dwellings_occupancy_period_nuts.csv')

    #construction period distribution of unoccupied dwellings
    arch = df.stack().unstack(['yr_con', 'bld_age'])
    arch = arch.div(arch.sum(1), axis=0)
    arch = arch.stack(['yr_con', 'bld_age']).unstack(-3)
    arch['Occupied conventional dwellings'] = arch['Occupied conventional dwellings'].fillna(arch.mean()['Occupied conventional dwellings'])
    arch['Unoccupied conventional dwellings'] = arch['Unoccupied conventional dwellings'].fillna(arch.mean()['Unoccupied conventional dwellings'])
    arch['value'] = arch['Unoccupied conventional dwellings'].round(3)
    arch = arch.drop(['Occupied conventional dwellings','Unoccupied conventional dwellings'], axis=1).unstack(['yr_con', 'bld_age'])
    arch.loc[:,('value', 1945, 'p1')] = arch[('value', 1945, 'p1')].sub(arch.sum(1).sub(1))
    arch = arch.stack(['yr_con', 'bld_age']).reset_index()
    arch = arch.reindex(columns=['region_bld', 'region_gea','urt','clim','mat', 'arch', 'yr_con', 'bld_age', 'value'])
    arch.to_csv('data/input_csv_NUTS_2025_resid/shr_vacant_base_period_resid.csv', index=False)

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_61064/2846942287.py:5: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  df = df.set_index(df.columns.drop(['Occupied conventional dwellings','Unoccupied conventional dwellings']).to_list()).mul(dwellings.groupby(level=[0,1,2], axis=0).sum()).groupby(level=['region_bld', 'clim', 'region_gea', 'urt', 'mat', 'arch', 'yr_con', 'bld_age'], axis=0).sum()
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_61064/2846942287.py:5: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  df = df.set_index(df.columns.drop(['Occupied conventional dwellings','Unoccupied conventional dwellings']).to_list()).mul(dwellings.groupby(level=[0,1,2], axis=0).sum()).groupby(level=['region_bld', 'clim', 'region_gea', 'urt', 'mat', 'arch', 'yr_con', 'bld_age'], axis=0).sum()
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipyke

### Disaggregating non-specified inputs

In [13]:
#NOTE: values for shr_need and bld_mat are only binary (0,1) in Europe in original files
def detail_other_inputs(idx_full):
    # Non-specified Inputs Detailed: expanding tables to cover all NUTS regions
    column_list = ['region_nuts']
    input_list = pd.read_csv('data/input_list_resid_2025_11_06_nuts.csv')['SSP\xa02.00'].to_list()
    replacement_map = {#inc_cl
                    'q1':1,
                    'q2':2,
                    'q3':3,
                    #bld_age
                    'ns':0,
                    'p1':1,
                    'p2':2,
                    'p3':3,
                    #eneff
                    'ns':0, 
                    's1':1, 
                    's2':2, 
                    's3':3, 
                    'sr11_std':4,
                    'sr21_std':5, 
                    'sr31_std':6, 
                    's51_std':7, 
                    'sr12_low':8,
                    'sr22_low':9,  
                    'sr32_low':10,
                    's52_low':11, 
                    #clim
                    'Zone_0A Extremely hot Humid':0, 
                    'Zone_1A Very hot Humid':1,
                    'Zone_2A Hot Humid':2, 
                    'Zone_2B Hot Dry':2.5, 
                    'Zone_3A Warm Humid':3,
                    'Zone_3B Warm Dry':3.3, 
                    'Zone_3C Warm Marine':3.7, 
                    'Zone_4A Mixed Humid':4,
                    'Zone_4C Mixed Marine':4.5,
                    'Zone_5A Cool Humid':5,
                    'Zone_5C Cool Marine':5.5, 
                    'Zone_6A Cold Humid':6,
                    'Zone_7A Very cold Humid':7, 
                    'Zone_8A Subarctic/arctic Humid':8,
                    #arch
                    'mfh':0,
                    'sfh':1,
                    'inf':2,
                    np.inf:2,
                    #mat
                    'sub':1,
                    'perm':0,
                    #region_gea
                    'WEU':0,
                    'EEU':1,
                    #urt
                    'rur':0,
                    'sub':1,
                    'urb':2,
                    }
    for filename in input_list:
        if filename not in ['regions_R61','climatic_zones_rev', 'bld_shr_arch_resid', 'stock_baseyear_resid_rev_share', 'stock_baseyear_resid_rev', 'pop_clim_rev_SSP2', 'hh_size_rev', 'floor_resid_ssp2_rev']: #for filename in ['heat_intensity_rev', 'cool_intensity_rev', 'cool_days_rev', 'shr_need_cool_resid_rev', 'shr_need_heat_resid_rev', 'bld_shr_access_cool_resid_ssp2_rev', 'bld_share_mat_resid_ssp2_rev']:
            input = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')
            if (urbanity == True) and ('urt' in input.columns):
                if 'value' in input.columns:
                    input = input.set_index(input.columns.drop('value').to_list()).unstack('urt')
                    input[('value','sub')] = input.mean(1)
                    input = input.stack().reset_index()
                else:
                    input = pd.concat([input,input[input['urt']=='urb'].replace('urb', 'sub')])
            if (('region_bld' in input.columns) and ('clim' in input.columns)): #there are two more files which have bld but not clim, households and floor space, those are disaggregated below
                if 'region_bld' in input.columns:
                    region_agg = 'region_bld'
                else:
                    region_agg = 'region_gea'
                input_cols = list(input.columns.drop('value'))
                print(input_cols)
                #merge files
                idx_merge = idx_full[input_cols + ['region_nuts']].drop_duplicates()
                input_nuts = pd.merge(input, idx_merge, on=input_cols, how='right')
                input_nuts_cols = list(input_nuts.columns.drop('value'))
                df = input_nuts.set_index(input_nuts_cols)

                #numerical encoding of ordinal, binary and categorical data
                input_nuts = input_nuts.replace(replacement_map)
                input_nuts = pd.concat([input_nuts, pd.get_dummies(input_nuts[region_agg], dtype=float)], axis=1)
                input_nuts = input_nuts.drop([region_agg, 'region_nuts'], axis=1)

                #impute values for missing country-climate zone combinations based on nearest neighbours
                input_nuts = pd.DataFrame(KNNImputer(missing_values=np.nan, n_neighbors=3, weights='uniform')\
                            .fit(input_nuts).transform(input_nuts), index=df.index, columns=input_nuts.columns)['value'].reset_index()

                input_nuts['value'] = round(input_nuts.value, 8)
                
                #export
                input_nuts = input_nuts[~input_nuts['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
                input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv', index=False)

                #checks
                print(input_nuts_cols)
                original = input[input[region_agg].isin(region_nuts_lab[region_agg].unique())]
                out = input_nuts.copy()
                print('Missing values: ',out.isna().sum().sum())
                print('Null before: ',len(original[original.value==0]))
                print('Null after: ',len(out[out.value==0]))
                print('Duplicates: ',out.duplicated().sum().sum())
                print('NUTS3 regions: ',len(out.region_nuts.unique()))
                print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values(region_agg).reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values(region_agg).reset_index(drop=True)))

                
                print(filename + ' DONE')
    
    return column_list

idx_full = pd.read_csv('/Volumes/KINGSTON/data/idx_full.csv') #takes 3 minutes
if urbanity == True:
    idx_full = pd.concat([idx_full,idx_full[idx_full['urt']=='urb'].replace('urb', 'sub')])
detail_other_inputs(idx_full) #takes 50 minutes

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_20251/3996330836.py:114: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  idx_full = pd.read_csv('/Volumes/KINGSTON/data/idx_full.csv') #takes 3 minutes
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_20251/3996330836.py:66: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  input = input.stack().reset_index()
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_20251/3996330836.py:66: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  input = input.sta

['region_bld', 'clim', 'inc_cl', 'mat', 'year', 'urt']


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_20251/3996330836.py:83: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  input_nuts = input_nuts.replace(replacement_map)


['region_bld', 'clim', 'inc_cl', 'mat', 'year', 'urt', 'region_nuts']
Missing values:  0
Null before:  8262
Null after:  188730
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True
bld_share_mat_resid_ssp2_rev DONE
['region_bld', 'clim', 'inc_cl', 'year', 'urt']


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_20251/3996330836.py:66: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  input = input.stack().reset_index()
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_20251/3996330836.py:83: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  input_nuts = input_nuts.replace(replacement_map)


...

['region_bld', 'clim', 'arch', 'eneff', 'urt']


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_20251/3996330836.py:83: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  input_nuts = input_nuts.replace(replacement_map)


['region_bld', 'clim', 'arch', 'eneff', 'urt', 'region_nuts']
Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True
shr_need_heat_resid_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_20251/3996330836.py:66: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  input = input.stack().reset_index()
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_20251/3996330836.py:66: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  input = input.stack().reset_index()


['region_nuts']

In [ ]:
                # Heat Operation Hours (only input with region_gea + clim differentiation)

                filename = 'heat_operation_hours_ssp2'
                input = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')

                region_nuts_clim_year = pd.concat([region_nuts_clim]*len(input.year.unique()), keys=list(input.year.unique()), names=['year', 'index']).reset_index().drop('index', axis=1)
                region_nuts_clim_year['region_gea'] = region_nuts_clim_year.region_bld.str[2:5]
                region_nuts_clim_year.drop(['urt'], axis=1, inplace=True)
                region_nuts_clim_year.drop_duplicates(inplace=True)

                input_nuts = pd.merge(input, region_nuts_clim_year, how='right')


                input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv', index=False)

                #checks
                original = input[input.region_gea.isin(['WEU', 'EEU'])]
                out = input_nuts.copy()
                print('Missing values: ',out.isna().sum().sum())
                print('Null before: ',len(original[original.value==0]))
                print('Null after: ',len(out[out.value==0]))
                print('Duplicates: ',out.duplicated().sum().sum())
                print('NUTS3 regions: ',len(out.region_nuts.unique()))
                print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_gea').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop(['region_nuts', 'region_bld'], axis=1).drop_duplicates().sort_values('region_gea').reset_index(drop=True)))

                print(filename + ' DONE')


                for filename in ['material_int_resid', 'bld_share_fuel_heat_resid', 'shr_hh_tenr', 'bld_shr_district_heat_resid', 'bld_share_mat_resid_ssp2_rev', 'discount_rate_new', 'discount_rate_ren', 'ct_bld', 'ct_inc_cl']:
                    input = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')
                    if (urbanity == True) and ('urt' in input.columns):
                        if 'value' in input.columns:
                            input = input.set_index(input.columns.drop('value').to_list()).unstack('urt')
                            input[('value','sub')] = input.mean(1)
                            input = input.stack().reset_index()
                        else:
                            input = pd.concat([input,input[input['urt']=='urb'].replace('urb', 'sub')])
                        
                        input.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv', index=False)

Missing values:  0
Null before:  216
Null after:  72
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  False
heat_operation_hours_ssp2 DONE


### Adding NUTS region specific data: household sizes, floor area, population

In [8]:
# Household size: current

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWBNO_R3
input = pd.read_csv('data/input_detailing_NUTS/estat_cens_21dwbno_r3_en.csv', usecols=['n_person', 'Type of building','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input.replace({'GE11':'12'}, inplace=True) #aligning labels with model
input = input.loc[~input['n_person'].isin(['3-5', '6-10', 'GE6', 'TOTAL'])] #only direct numbers
input['population'] = input['n_person'].astype(float) * input['OBS_VALUE'] #calculate population per NUTS3 region based on number of occupants per dwelling and number of dwellings
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch']).sum(numeric_only=True)
input['hh_size'] = input['population'] / input['OBS_VALUE'] #calculate average household size per NUTS3 region and type of building based on population divided by number of dwellings
input = input.reset_index().drop(['OBS_VALUE'], axis=1).rename({'geo':'NUTS-3 Code'}, axis=1)

input = nuts3code_to_region_nuts(input)

input = input.groupby(by=['region_bld','region_nuts', 'arch']).agg({'population':'sum', 'hh_size':'mean'})

input_hh = input.copy()

# household size: trend
filename = 'hh_size_rev'
input_trend_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')
if urbanity == True:
       input_trend_original = pd.concat([input_trend_original,input_trend_original[input_trend_original['urt']=='urb'].replace('urb', 'sub')]).sort_values('region_bld')
input_trend_original = pd.concat([input_trend_original.set_index(['region_bld', 'urt', 'year']), input_trend_original.set_index(['region_bld', 'urt', 'year'])], keys=['mfh', 'sfh'], names=['arch','region_bld', 'urt', 'year']).reorder_levels([1,2,0,3])

if urbanity == True:
    #only for national recalibration, not for NUTS regions since dwelling stock by urbanity and NUTS region not provided
    #calculate urbanity share in dwellings
    pop_by_arch_degurba = pd.read_csv('data/input_detailing_NUTS/ilc_lvho01_page_linear.csv', usecols=['building', 'deg_urb','geo', 'OBS_VALUE'], index_col=[0,1,2])
    hhsize_by_degurba = pd.read_csv('data/input_detailing_NUTS/hbs_car_t315_page_linear.csv', usecols=['deg_urb','geo', 'OBS_VALUE'], index_col=[0,1])
    pop_by_degurba = pop_by_arch_degurba.xs('Total', axis=0, level=0).drop('Total', axis=0, level=0)
    dw_by_degurba = pop_by_degurba.div(hhsize_by_degurba)
    dw_by_degurba.update(pop_by_degurba, overwrite=False)
    dw_by_degurba = dw_by_degurba.unstack('deg_urb')
    dw_by_degurba = dw_by_degurba.div(dw_by_degurba.sum(1), axis=0).stack(1).reset_index()
    dw_by_degurba['iso3'] = coco.convert(names=dw_by_degurba['geo'], to='ISO3')
    dw_by_degurba = pd.merge(dw_by_degurba, region_lab[['region_bld', 'iso3']], on='iso3',how='left').dropna(subset='region_bld').replace({'Cities':'urb', 'Towns and suburbs':'sub', 'Rural areas':'rur'})
    dw_by_degurba = dw_by_degurba.rename({'deg_urb':'urt', 'OBS_VALUE':'value'}, axis=1)
    dw_by_degurba = dw_by_degurba[['region_bld', 'urt', 'value']]
    dw_by_degurba = dw_by_degurba.set_index(dw_by_degurba.columns.drop('value').to_list())
    
    #calculate building type share in dwellings
    pop_by_arch_degurba = pd.read_csv('data/input_detailing_NUTS/ilc_lvho01_page_linear.csv', usecols=['building', 'deg_urb','geo', 'OBS_VALUE'], index_col=[0,1,2])
    dw_by_arch_degurba = pop_by_arch_degurba.div(hhsize_by_degurba)
    dw_by_arch_degurba.update(pop_by_arch_degurba, overwrite=False)
    dw_by_arch_degurba = dw_by_arch_degurba.drop(['Others', 'Total', np.nan], axis=0, level=0).drop('Total', axis=0, level=1).unstack('building')
    dw_by_arch_degurba = dw_by_arch_degurba.div(dw_by_arch_degurba.sum(1), axis=0).stack(1).reset_index()
    dw_by_arch_degurba['iso3'] = coco.convert(names=dw_by_arch_degurba['geo'], to='ISO3')
    dw_by_arch_degurba = pd.merge(dw_by_arch_degurba, region_lab[['region_bld', 'iso3']], on='iso3',how='left').dropna(subset='region_bld').replace({'Cities':'urb', 'Towns and suburbs':'sub', 'Rural areas':'rur', 'Flat':'mfh', 'House':'sfh'})
    dw_by_arch_degurba = dw_by_arch_degurba.rename({'deg_urb':'urt', 'building':'arch', 'OBS_VALUE':'value'}, axis=1)
    dw_by_arch_degurba = dw_by_arch_degurba[['region_bld', 'urt', 'arch', 'value']]
    dw_by_arch_degurba = dw_by_arch_degurba.set_index(dw_by_arch_degurba.columns.drop('value').to_list())

    #reformat hhsize by urbanity
    hhsize_by_degurba = hhsize_by_degurba.drop(['Unknown', 'Total', np.nan], axis=0, level='deg_urb')
    hhsize_by_degurba = hhsize_by_degurba.reset_index()
    hhsize_by_degurba['iso3'] = coco.convert(names=hhsize_by_degurba['geo'], to='ISO3')
    hhsize_by_degurba = pd.merge(hhsize_by_degurba, region_lab[['region_bld', 'iso3']], on='iso3',how='left').dropna(subset='region_bld').replace({'Cities':'urb', 'Towns and suburbs':'sub', 'Rural areas':'rur'})
    hhsize_by_degurba = hhsize_by_degurba.rename({'deg_urb':'urt', 'OBS_VALUE':'value'}, axis=1)
    hhsize_by_degurba = hhsize_by_degurba[['region_bld', 'urt', 'value']]
    hhsize_by_degurba = hhsize_by_degurba.set_index(hhsize_by_degurba.columns.drop('value').to_list())

    #calculate household size by arch and urbanity
    hhsize_in_degurba_data = ((dw_by_arch_degurba * dw_by_degurba).div((dw_by_arch_degurba * dw_by_degurba).groupby(axis=0, level=['region_bld', 'arch']).sum())).mul(hhsize_by_degurba).groupby(level=['region_bld', 'arch']).sum()
    hhsize_by_arch_degurba = (hhsize_by_degurba / hhsize_in_degurba_data).unstack('urt').dropna(axis=1, how='all').fillna(1).mul(input_trend_original.groupby(level=['region_bld','arch']).mean()['value'], axis=0).stack('urt')

    #update input_trend_original to include urbanity and building type variation in household sizes
    for i,value in input_trend_original.iterrows():
        region, urt, arch, year = i
        try:
            input_trend_original.loc[i,'value'] = hhsize_by_arch_degurba.loc[(region, arch, urt), 'value']
        except:
            input_trend_original.loc[i,'value'] = input_trend_original.loc[i,'value']
    #export csv
    input_trend_original.reset_index().to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_sub.csv', index=False)

#replace missing values in EUROSTAT with original model data
input_trend = pd.merge(input_trend_original.reset_index(), region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1), how='right')
input_trend['region_nuts'] = input_trend['region_nuts'].fillna(input_trend['region_bld'])

#include input['fa_pc'] values fom EUROSTAT
input_trend_regurtarch = input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'arch']).mean(numeric_only=True).drop('year', axis=1)
input_trend_regurtarch.update(input.replace(0,np.nan).rename({'hh_size':'value'}, axis=1).drop('population', axis=1))

#expand fa_pc by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'arch', 'year']).div(input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'arch']).mean(numeric_only=True).drop('year', axis=1)) #QQ: why is there only hhd size trend in Croatia and no other EU countries?
#input_trend_diff = input_trend_diff.assign(value=1) #replacing Croatian trend in household sizes with stagnant household sizes
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() 

#export csv
input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv', index=False)

#checks
original = input_trend_original.reset_index()[input_trend_original.reset_index().region_bld.isin(region_nuts_lab.region_bld.unique())]
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_30286/2097457459.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  input = pd.read_csv('data/input_detailing_NUTS/estat_cens_21dwbno_r3_en.csv', usecols=['n_person', 'Type of building','geo', 'OBS_VALUE'])


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [9]:
# Floor area: current

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWBNR_R3
input = pd.read_csv('data/input_detailing_NUTS/estat_cens_21dwbnr_r3_filtered_en.csv', usecols=['area', 'Type of building','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['area'].isin(['UNK'])] #only direct numbers
input.replace({'SQM_LT30':20, 'SQM30-39':35,'SQM40-49':45,'SQM50-59':55,'SQM60-79':70,'SQM80-99':90, 'SQM100-119':110, 'SQM120-149':135, 'SQM_GE150':200}, inplace=True) #aligning labels with model
input['fa_total'] = input['area'].astype(float) * input['OBS_VALUE'] #calculate population per NUTS3 region based on number of occupants per dwelling and number of dwellings
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch']).sum()
input = input.reset_index().drop(['OBS_VALUE', 'area'], axis=1).rename({'geo':'NUTS-3 Code'}, axis=1)
input = nuts3code_to_region_nuts(input)
input = input.groupby(by=['region_bld','region_nuts', 'arch']).sum()
input['fa_pc'] = input['fa_total'] / input_hh['population'] #calculate average household size per NUTS3 region and type of building based on population divided by number of dwellings

#fill up missing values in input['fa_pc']
# floor area: trend and missing values
filename = 'floor_resid_ssp2_rev'
input_trend_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')
if urbanity == True:
       input_trend_original = pd.concat([input_trend_original,input_trend_original[input_trend_original['urt']=='urb'].replace('urb', 'sub')]).sort_values('region_bld')
       m2_by_degurba = pd.read_csv('data/input_detailing_NUTS/ilc_lvho31$defaultview_linear.csv', usecols=['deg_urb','geo', 'OBS_VALUE'], index_col=[0,1])
       m2_by_degurba = m2_by_degurba.drop('Total', axis=0, level='deg_urb')
       hhsize_by_degurba = pd.read_csv('data/input_detailing_NUTS/hbs_car_t315_page_linear.csv', usecols=['deg_urb','geo', 'OBS_VALUE'], index_col=[0,1])
       hhsize_by_degurba.drop(['Unknown', 'Total', np.nan], axis=0, level='deg_urb')
       m2pc_by_degurba = m2_by_degurba.div(hhsize_by_degurba, fill_value=2.3425)
       m2pc_by_degurba = m2pc_by_degurba[m2pc_by_degurba['OBS_VALUE'] > 10].reset_index()
       m2pc_by_degurba['iso3'] = coco.convert(names=m2pc_by_degurba['geo'], to='ISO3')
       m2pc_by_degurba = pd.merge(m2pc_by_degurba, region_lab[['region_bld', 'iso3']], on='iso3',how='left').dropna(subset='region_bld').replace({'Cities':'urb', 'Towns and suburbs':'sub', 'Rural areas':'rur'})
       m2pc_by_degurba = m2pc_by_degurba.rename({'deg_urb':'urt', 'OBS_VALUE':'value'}, axis=1)
       m2pc_by_degurba = m2pc_by_degurba[['region_bld', 'urt', 'value']]
       m2pc_by_degurba = m2pc_by_degurba.set_index(m2pc_by_degurba.columns.drop('value').to_list())

       #calculate household size by arch and urbanity
       m2pc_in_degurba_data = ((dw_by_arch_degurba * dw_by_degurba).div((dw_by_arch_degurba * dw_by_degurba).groupby(axis=0, level=['region_bld', 'arch']).sum())).mul(m2pc_by_degurba).groupby(level=['region_bld', 'arch']).sum()
       input_trend_original = input_trend_original.set_index(input_trend_original.columns.drop('value').to_list())
       m2pc_by_arch_degurba = (m2pc_by_degurba / m2pc_in_degurba_data).unstack('urt').dropna(axis=1, how='all').fillna(1).mul(input_trend_original.groupby(level=['region_bld','arch', 'year']).mean()['value'], axis=0).stack('urt')

       #update input_trend_original to include urbanity and building type variation in household sizes
       for i,value in input_trend_original.iterrows():
              region, urt, arch, mat, year = i
              try:
                     input_trend_original.loc[i,'value'] = m2pc_by_arch_degurba.loc[(region, arch, year, urt), 'value']
              except:
                     input_trend_original.loc[i,'value'] = input_trend_original.loc[i,'value']
       
       input_trend_original = input_trend_original.reset_index()
       
       #export csv
       input_trend_original.reset_index().to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_sub.csv', index=False)
       
#replace missing values in EUROSTAT with original model data
input_trend = pd.merge(input_trend_original, region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1), how='right')

#include input['fa_pc'] values fom EUROSTAT
input_trend_regurtarch = input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'arch']).mean(numeric_only=True).drop('year', axis=1)
input_trend_regurtarch.update(input.replace(0,np.nan).rename({'fa_pc':'value'}, axis=1))

#expand fa_pc by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'mat','arch', 'year']).div(input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'arch']).mean(numeric_only=True).drop('year', axis=1))
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() #QQ: why is there a twice as high floor arae per capita for the latest energy efficiency standards?

input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv', index=False)

#checks
original = input_trend_original[input_trend_original.region_bld.isin(region_nuts_lab.region_bld.unique())]
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_30286/1114755535.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  input.replace({'SQM_LT30':20, 'SQM30-39':35,'SQM40-49':45,'SQM50-59':55,'SQM60-79':70,'SQM80-99':90, 'SQM100-119':110, 'SQM120-149':135, 'SQM_GE150':200}, inplace=True) #aligning labels with model


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [10]:
# Population: current
#use higher resolution projection in case 'suburban' dimension is required: coclico reprojection of IIASA SSP2
filename = 'pop_clim_rev_SSP2'
if urbanity == True:
    pop_by_nuts_degurba = pd.read_csv('data/input_detailing_NUTS/pop_by_nuts_degurba.csv')
    pop_by_nuts_degurba = pop_by_nuts_degurba.rename({'NUTS_ID':'NUTS-3 Code'}, axis=1, level=0).groupby(by=['NUTS-3 Code', 'urt', 'year']).sum(numeric_only=True).drop('degurba', axis=1).reset_index()
    pop_by_nuts_degurba = nuts3code_to_region_nuts(pop_by_nuts_degurba)
    pop_by_nuts_degurba = pd.merge(pop_by_nuts_degurba, region_nuts_clim.drop('urt', axis=1).drop_duplicates()) #adding climate zone categories
    pop_by_nuts_degurba.rename({'population':'value'}, axis=1, inplace=True)
    pop_by_nuts_degurba = pop_by_nuts_degurba.drop_duplicates().set_index(pop_by_nuts_degurba.columns.drop('value').to_list()).unstack('year').droplevel(0, axis=1).reindex(list(range(2010,2101,5)),axis=1).interpolate('cubic', axis=1)
    pop_by_nuts_degurba = pop_by_nuts_degurba.stack('year').div(1e6)
    pop_by_nuts_degurba = pop_by_nuts_degurba.unstack('urt').fillna(0).stack('urt').reset_index().rename({0:'value'}, axis=1)
    input_trend_out = pop_by_nuts_degurba.reindex(['region_bld','region_nuts', 'urt', 'clim', 'year', 'value'], axis=1)
    
    #for regions outside the European continent import EUROSTAT projections: https://doi.org/10.2908/PROJ_19RP3
    input = pd.read_csv('data/input_detailing_NUTS/estat_proj_19rp3_filtered_en.csv', usecols=['geo', 'TIME_PERIOD','OBS_VALUE']) #residents per territory and year
    input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
    input = input.rename({'geo':'NUTS-3 Code'}, axis=1)
    input = nuts3code_to_region_nuts(input)
    input = pd.merge(input, region_nuts_clim.drop('urt', axis=1).drop_duplicates()) #adding climate zone categories
    input.rename({'TIME_PERIOD':'year', 'OBS_VALUE':'value'}, axis=1, inplace=True)
    input = input[input['year'].isin(list(range(2020,2101,5)))]
    input = (input.groupby(by=['region_bld','region_nuts', 'clim', 'year']).sum()/1e6) #million residents in each year
    input = input.reset_index()
    input = input[input.region_nuts.isin(list(set(region_nuts_lab.region_nuts) - set(input_trend_out.region_nuts)))]
    input_null = input.copy()
    input_null['value'] = 0
    missing_regions = pd.concat([input_null, input, input_null], keys=['rur', 'sub', 'urb']).reset_index().drop('level_1', axis=1).rename({'level_0':'urt'}, axis=1)
    input_trend_out = pd.concat([input_trend_out, missing_regions])
else:
    #import EUROSTAT projections: https://doi.org/10.2908/PROJ_19RP3
    input = pd.read_csv('data/input_detailing_NUTS/estat_proj_19rp3_filtered_en.csv', usecols=['geo', 'TIME_PERIOD','OBS_VALUE']) #residents per territory and year
    input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
    input = input.rename({'geo':'NUTS-3 Code'}, axis=1)
    input = nuts3code_to_region_nuts(input)
    input = pd.merge(input, region_nuts_clim.drop('urt', axis=1).drop_duplicates()) #adding climate zone categories
    input.rename({'TIME_PERIOD':'year', 'OBS_VALUE':'value'}, axis=1, inplace=True)
    input = input[input['year'].isin(list(range(2020,2101,5)))]
    input = (input.groupby(by=['region_bld','region_nuts', 'clim', 'year']).sum()/1e6) #million residents in each year


    #population national distributio
    input_trend_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv') #million residents per country, urbanity, climate zone, and year

    #merge index by duplicating values in original model data
    region_nuts_clim_year = pd.concat([region_nuts_clim]*len(input_trend_original.year.unique()), keys=list(input_trend_original.year.unique()), names=['year', 'index']).reset_index().drop('index', axis=1)
    input_trend = pd.merge(input_trend_original, region_nuts_clim_year, how='right') #duplicating population across same urt, clim, country
    input_trend = input_trend.set_index(['region_bld', 'region_nuts', 'urt', 'clim', 'year'])

    #fill those regions for which no matching country-climate combinations were available, using simply the matching country-urbanity combination
    input_trend_mean = input_trend.groupby(axis=0, level=[0,2,4]).mean() #country, urbanity, year used for filling nan values
    filler_mapped = input_trend_mean.reindex(input_trend.index.droplevel([1, 3])) #nuts and climate not used
    filler_mapped.index = input_trend.index
    input_trend = input_trend.fillna(filler_mapped).reset_index()

    #normalise population assuming equal shares of population across all nuts-urt combinations within each country-climate combination
    nuts_in_reg = region_nuts_clim.drop('urt', axis=1).drop_duplicates()[['region_bld', 'clim']].value_counts().reset_index() #number of nuts within each country-climate combination
    nuts_in_reg.columns = ['region_bld', 'clim', 'value']
    input_trend = input_trend.set_index(['region_bld','region_nuts', 'urt', 'clim', 'year']).div(nuts_in_reg.set_index(['region_bld', 'clim'])).reset_index() #normalise population assuming equal shares of population across all nuts-urt combinations within each country-climate combination

    #include input values fom EUROSTAT
    input_trend_regurtarch = input_trend.groupby(by=['region_bld','region_nuts', 'clim', 'year']).sum(numeric_only=True) #population per nuts and year
    input_trend_regurtarch.update(input.replace(0,np.nan)) #

    #expand pop by using distribution of values from input_trend_original
    input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'clim', 'year']).div(input_trend.groupby(by=['region_bld','region_nuts', 'clim', 'year']).sum(numeric_only=True))
    input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() 

input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv', index=False)

#checks
original = input_trend_original[input_trend_original.region_bld.isin(region_nuts_lab.region_bld.unique())]
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_30286/1551907971.py:51: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_trend_mean = input_trend.groupby(axis=0, level=[0,2,4]).mean() #country, urbanity, year used for filling nan values


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [30]:
#splitting the population projection up in three files: total population, urbanity shares, climate zone shares

filename = 'pop_clim_rev_SSP2'
pop = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')

pop_agg = pop.groupby(by=['region_bld', 'region_nuts', 'year']).sum().reset_index().drop(['urt', 'clim'], axis=1)
pop_agg.to_csv('data/input_csv_NUTS_2025_resid/R61_pop_ssp2_2026-01-29'+resolution+'.csv', index=False)

pop_clim = pop.drop(['year', 'value'], axis=1)
pop_clim['value'] = 1 #since each NUTS region is directly matched with one climatic zone, the share of the climatic zone in the NUTS region is always 1
pop_clim.to_csv('data/input_csv_NUTS_2025_resid/pop_shr_climatic_zones_R61_2026-01-29'+resolution+'.csv', index=False)

pop_urt = pop.set_index(pop.drop('value', axis=1).columns.to_list()).unstack('urt')
pop_urt = pop_urt.div(pop_urt.sum(1), axis=0)
pop_urt = pop_urt.stack().reset_index()
pop_urt = pop_urt[['region_bld', 'region_nuts','year','urt', 'value']]
pop_urt.to_csv('data/input_csv_NUTS_2025_resid/R61_urt_ssp2_2026-01-29'+resolution+'.csv', index=False)

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_32482/1075915924.py:15: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  pop_urt = pop_urt.stack().reset_index()


## Decrease resolution from NUTS3 to national BLD aggregate level

In [ ]:
#Check which files contain na values
import os
import glob

folder_path = "/Users/Mira/Desktop/MCC/Code/EUBUCCO-IAM/message-ix-buildings/message_ix_buildings/sturm/data/input_csv_NUTS_2025_resid"#"/Users/Mira/Desktop/MCC/Code/EUBUCCO-IAM/messageix-buildings-subnational/2025_EU/input_resid"
files_with_urt = [os.path.basename(f) for f in glob.glob(os.path.join(folder_path, "*.csv"))] #if '_sub' in os.path.basename(f)
print("Files with '_sub' column:")
for filename in sorted(files_with_urt):
    if pd.read_csv(folder_path+"/"+filename).isnull().any().any():
        print(f"  {filename}")

Files with '_sub' column:
  stock_baseyear_resid_rev_share_unoccupied_nuts.csv
  stock_baseyear_resid_rev_share_unoccupied_nuts_sub.csv


In [ ]:
#Check which files contain duplicates
import os
import glob
"""
files containing duplicates for '_nuts' and for '_nuts_sub':
shr_need_heat_resid_rev_nuts.csv
shr_need_cool_resid_rev_nuts.csv
hr_need_cool_resid_rev_nuts_sub.csv
cool intensity
heat intensity
"""
folder_path = "/Users/Mira/Desktop/MCC/Code/EUBUCCO-IAM/message-ix-buildings/message_ix_buildings/sturm/data/input_csv_NUTS_2025_resid"#"/Users/Mira/Desktop/MCC/Code/EUBUCCO-IAM/messageix-buildings-subnational/2025_EU/input_resid"
files_with_urt = [os.path.basename(f) for f in glob.glob(os.path.join(folder_path, "*.csv"))] #if '_sub' in os.path.basename(f)
print("Files with '_sub' column:")
for filename in sorted(files_with_urt):
    input = pd.read_csv(folder_path+"/"+filename)
    print(filename, input.shape)
    print(filename, input.drop_duplicates().shape)

Files with '_sub' column:
bld_demolition_distr_long.csv (198, 6)
bld_demolition_distr_long.csv (198, 6)
bld_lifetime_new.csv (11, 2)
bld_lifetime_new.csv (11, 2)
bld_lifetime_ren.csv (11, 2)
bld_lifetime_ren.csv (11, 2)
bld_share_fuel_heat_resid.csv (110, 4)
bld_share_fuel_heat_resid.csv (110, 4)
bld_share_fuel_heat_resid_nuts_sub.csv (165, 4)
bld_share_fuel_heat_resid_nuts_sub.csv (165, 4)
bld_share_fuel_heat_resid_nuts_sub_bld.csv (165, 4)
bld_share_fuel_heat_resid_nuts_sub_bld.csv (165, 4)
bld_share_fuel_heat_resid_nuts_sub_bld_flat.csv (55, 3)
bld_share_fuel_heat_resid_nuts_sub_bld_flat.csv (55, 3)
bld_share_mat_resid_ssp2_rev.csv (48816, 7)
bld_share_mat_resid_ssp2_rev.csv (48816, 7)
bld_share_mat_resid_ssp2_rev_nuts.csv (251640, 8)
bld_share_mat_resid_ssp2_rev_nuts.csv (251640, 8)
bld_share_mat_resid_ssp2_rev_nuts_bld.csv (16632, 7)
bld_share_mat_resid_ssp2_rev_nuts_bld.csv (16632, 7)
bld_share_mat_resid_ssp2_rev_nuts_sub.csv (73224, 7)
bld_share_mat_resid_ssp2_rev_nuts_sub.csv (

In [31]:
sub = pd.read_csv(folder_path+"/"+"shr_need_cool_resid_rev_nuts_sub_bld.csv")
sub.loc[sub['value']>1, 'value'] = sub['value']/2
sub['value'] = sub['value'].round(5)
sub.to_csv(folder_path+"/"+"shr_need_cool_resid_rev_nuts_sub_bld.csv")

sub = pd.read_csv(folder_path+"/"+"shr_need_heat_resid_rev_nuts_sub_bld.csv")
sub.loc[sub['value']>1, 'value'] = sub['value']/2
sub['value'] = sub['value'].round(5)
sub.to_csv(folder_path+"/"+"shr_need_heat_resid_rev_nuts_sub_bld.csv")

In [69]:
sub = pd.read_csv(folder_path+"/"+"stock_baseyear_resid_rev_share_nuts_sub_bld.csv")
sub.groupby(sub.columns.drop(['yr_con', 'bld_age', 'value']).to_list()).sum(numeric_only=True).value == 0
sub = sub.set_index(sub.columns.drop(['value']).to_list())
sub.loc[sub.groupby([item for item in list(sub.index.names) if item not in ['yr_con', 'bld_age', 'value']]).sum(numeric_only=True).value == 0, 'value'] = 1/14 #len(sub.yr_con.unique())
sub.reset_index().to_csv(folder_path+"/"+"shr_need_heat_resid_rev_nuts_sub_bld.csv")

input_list
- 'regions_R61', #no change necessary
- 'climatic_zones_rev' #no change necessary
- 'pop_clim_rev_SSP2' #no change necessary

- 'hh_size_rev' #mean weighted by number of dwellings (= pop / hhsize * arch_share) in each cell with arch weighted by bld_arch_share; by 'urt', 'arch', 'year'
 'floor_resid_ssp2_rev', #mean weighted by number of dwellings (=pop/hhsize*arch_share), ignore mat, always same; 'urt', 'mat', 'arch', 'year',
 'bld_shr_arch_resid', #mean weighted by number of dwellings (=pop/hhsize*arch_share), ignore mat, always same; 'urt', 'mat', 'arch', 'year',
 'stock_baseyear_resid_rev_share' #mean weighted by number of dwellings

- 'bld_share_mat_resid_ssp2_rev', #use original files, since no primary data injected
 'bld_shr_access_cool_resid_ssp2_rev', #use original files, since no primary data injected
 'heat_intensity_rev', #use original files, since no primary data injected
 'cool_intensity_rev', #use original files, since no primary data injected
 'cool_days', #use original files, since no primary data injected
 'shr_need_cool_resid_rev', #use original files, since no primary data injected
 'shr_need_heat_resid_rev', #use original files, since no primary data injected


In [6]:
    #Aggregating NUTS3 files back to NUTS0='region_bld'
    if resolution == '_nuts_sub':
        scenario = 'SSP2-NUTS-SUB'
    else:
        scenario = 'SSP2-NUTS'
    input_list = pd.read_csv('data/input_list_resid_2025_11_06_nuts.csv')[scenario].dropna().to_list()
    input_list = [i.removesuffix(resolution) for i in input_list]

    #calculating dwelling weights of dwellings in NUTS per country
    pop = pd.read_csv('data/input_csv_NUTS_2025_resid/pop_clim_rev_SSP2'+resolution+'.csv')
    pop['mat'] = 'perm'
    hhsize = pd.read_csv('data/input_csv_NUTS_2025_resid/hh_size_rev'+resolution+'.csv')
    hhsize['mat'] = 'perm'
    hhsize = hhsize.set_index(keys=hhsize.columns.drop('value').to_list())
    arch_shr = pd.read_csv('data/input_csv_NUTS_2025_resid/bld_shr_arch_resid'+resolution+'.csv')
    arch_shr = arch_shr.set_index(keys=arch_shr.columns.drop('value').to_list())

    dwellings = pop.groupby(by=pop.columns.drop(['clim','value']).to_list()).sum(numeric_only=True).mul(1e6).div((hhsize*arch_shr).groupby(level=[0,1,2,4,5]).sum()).fillna(method='bfill')
    dwellings = pd.merge(dwellings.reset_index(), region_nuts_clim)
    dwellings = dwellings.set_index(dwellings.columns.drop('value').to_list())

    dw_weights = dwellings.div(dwellings.groupby(level=['region_bld', 'urt', 'year', 'mat']).sum()).droplevel('clim')
    dw_weights_clim = dwellings.div(dwellings.groupby(level=['region_bld', 'urt', 'year', 'mat', 'clim']).sum())

    for filename in input_list:
        if (urbanity == True) and (filename in ['material_int_resid', 'bld_share_fuel_heat_resid', 'shr_hh_tenr', 'bld_shr_district_heat_resid', 'bld_share_mat_resid_ssp2_rev', 'discount_rate_new', 'discount_rate_ren', 'ct_bld', 'ct_inc_cl']):
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            input.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)
        elif filename in ['hh_size_rev','floor_resid_ssp2_rev', 'bld_shr_arch_resid']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            #mean weighted by dwellings per NUTS, rurality and year
            input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['stock_baseyear_resid_rev_share']:          
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            #mean weighted by dwellings per NUTS, rurality and year
            dw_weights_clim = dw_weights_clim.xs(2020, level='year')
            input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights_clim).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)
            dw_weights_clim = dwellings.div(dwellings.groupby(level=['region_bld', 'urt', 'year', 'mat', 'clim']).sum())

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                                        
            print(filename + ' DONE')
        elif filename not in ['regions_R61','climatic_zones_rev', 'pop_clim_rev_SSP2', 'hh_size_rev','floor_resid_ssp2_rev', 'bld_shr_arch_resid', 'stock_baseyear_resid_rev_share']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            #mean weighted by dwellings per NUTS, rurality and year
            if filename in ['heat_intensity_rev', 'cool_intensity_rev', 'cool_days', 'shr_need_cool_resid_rev', 'shr_need_heat_resid_rev']:
                dw_weights_ = dw_weights_clim.xs(2020, level='year')
            else:
                dw_weights_ = dw_weights_clim.copy()
            input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights_).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True)
            input_bld = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'.csv')
            input_bld = input_bld.set_index(input_bld.columns.drop('value').to_list())
            input_nuts.update(input_bld, overwrite=False)
            input_nuts = input_nuts.reset_index().drop_duplicates()
            if filename in ['shr_need_cool_resid_rev', 'shr_need_heat_resid_rev']:
                input_nuts.loc[input_nuts['value']>1, 'value'] = input_nuts['value']/2
                input_nuts['value'] = input_nuts['value'].round(5)
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['pop_clim_rev_SSP2']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['regions_R61','climatic_zones_rev']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            input_nuts = input.drop('region_nuts', axis=1).drop_duplicates()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)
                            
            print(filename + ' DONE')

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_58486/2045751871.py:18: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  dwellings = pop.groupby(by=pop.columns.drop(['clim','value']).to_list()).sum(numeric_only=True).mul(1e6).div((hhsize*arch_shr).groupby(level=[0,1,2,4,5]).sum()).fillna(method='bfill')


regions_R61 DONE
climatic_zones_rev DONE
Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
pop_clim_rev_SSP2 DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_58486/2045751871.py:93: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_58486/2045751871.py:33: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
hh_size_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_58486/2045751871.py:33: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()


...

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_58486/2045751871.py:71: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights_).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True)


Missing values:  0
Null before:  1040
Null after:  162
Duplicates:  0
shr_need_cool_resid_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_58486/2045751871.py:71: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights_).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True)


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
shr_need_heat_resid_rev DONE
Missing values:  0
Null before:  2202
Null after:  250
Duplicates:  0
stock_baseyear_resid_rev_share DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_58486/2045751871.py:50: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights_clim).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()


In [11]:
    #calculating dwelling weights of dwellings in NUTS per country
    pop = pd.read_csv('data/input_csv_NUTS_2025_resid/pop_clim_rev_SSP2'+resolution+'.csv')
    pop['mat'] = 'perm'
    hhsize = pd.read_csv('data/input_csv_NUTS_2025_resid/hh_size_rev'+resolution+'.csv')
    hhsize['mat'] = 'perm'
    hhsize = hhsize.set_index(keys=hhsize.columns.drop('value').to_list())
    arch_shr = pd.read_csv('data/input_csv_NUTS_2025_resid/bld_shr_arch_resid'+resolution+'.csv')
    arch_shr = arch_shr.set_index(keys=arch_shr.columns.drop('value').to_list())

    dwellings = pop.groupby(by=pop.columns.drop(['clim','value']).to_list()).sum(numeric_only=True).mul(1e6).div((hhsize*arch_shr).groupby(level=[0,1,2,4,5]).sum()).fillna(method='bfill')
    dwellings = pd.merge(dwellings.reset_index(), region_nuts_clim)
    dwellings = dwellings.set_index(dwellings.columns.drop('value').to_list())

    dw_weights = dwellings.div(dwellings.groupby(level=['region_bld', 'year', 'mat']).sum()).droplevel('clim')
    dw_weights_clim = dwellings.div(dwellings.groupby(level=['region_bld', 'year', 'mat']).sum())

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_49868/2325031681.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  dwellings = pop.groupby(by=pop.columns.drop(['clim','value']).to_list()).sum(numeric_only=True).mul(1e6).div((hhsize*arch_shr).groupby(level=[0,1,2,4,5]).sum()).fillna(method='bfill')


In [12]:
            filename = 'floor_resid_ssp2_rev'
            resolution = '_nuts'
            
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            dropcols = ['value', 'region_nuts']
            if 'clim' in input.columns.to_list():
                dropcols = dropcols + ['clim']
            if 'urt' in input.columns.to_list():
                dropcols = dropcols + ['urt']
            #mean weighted by dwellings per NUTS, rurality and year
            #input_nuts = 

In [ ]:
"""
I want to aggregate all NUTS parameterized values to BLD without variation by CLIM or URT
for this I need to aggregate 
1) all inputs with region BLD (no matter whether they have CLIM or URT)
but I do want to retain the index

potential issue: if i remove CLIM and URT from pop, the entire model will generate dwellings without those levels, might cause issues with the rest
"""

In [42]:
    #AGGREGATING TO BLD (without CLIM / URT)

    #Aggregating NUTS3 files back to NUTS0='region_bld'
    if resolution == '_nuts_sub':
        scenario = 'SSP2-NUTS-SUB'
    else:
        scenario = 'SSP2-NUTS'
    input_list = pd.read_csv('data/input_list_resid_2025_11_06_nuts.csv')[scenario].dropna().to_list()
    input_list = [i.removesuffix(resolution) for i in input_list]

    #calculating dwelling weights of dwellings in NUTS per country
    pop = pd.read_csv('data/input_csv_NUTS_2025_resid/pop_clim_rev_SSP2'+resolution+'.csv')
    pop['mat'] = 'perm'
    hhsize = pd.read_csv('data/input_csv_NUTS_2025_resid/hh_size_rev'+resolution+'.csv')
    hhsize['mat'] = 'perm'
    hhsize = hhsize.set_index(keys=hhsize.columns.drop('value').to_list())
    arch_shr = pd.read_csv('data/input_csv_NUTS_2025_resid/bld_shr_arch_resid'+resolution+'.csv')
    arch_shr = arch_shr.set_index(keys=arch_shr.columns.drop('value').to_list())

    dwellings = pop.groupby(by=pop.columns.drop(['clim','value']).to_list()).sum(numeric_only=True).mul(1e6).div((hhsize*arch_shr).groupby(level=[0,1,2,4,5]).sum()).fillna(method='bfill')
    dwellings = pd.merge(dwellings.reset_index(), region_nuts_clim)
    dwellings = dwellings.set_index(dwellings.columns.drop('value').to_list())

    dw_weights = dwellings.div(dwellings.groupby(level=['region_bld', 'year', 'mat']).sum()).droplevel('clim')
    dw_weights_clim = dwellings.div(dwellings.groupby(level=['region_bld', 'year', 'mat']).sum())

    for filename in input_list:
        if (urbanity == True) and (filename in ['material_int_resid', 'bld_share_fuel_heat_resid', 'shr_hh_tenr', 'bld_shr_district_heat_resid', 'bld_share_mat_resid_ssp2_rev', 'discount_rate_new', 'discount_rate_ren', 'ct_bld', 'ct_inc_cl']):
            #all inputs which contain urt but not region_bld, only region_gea
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            dropcols = []
            if 'clim' in input.columns.to_list():
                dropcols = dropcols + ['clim']
            if 'urt' in input.columns.to_list():
                dropcols = dropcols + ['urt']

            if 'value' in input.columns.to_list():
                dropcols = dropcols + ['value']
                input_nuts = input.set_index(input.columns.drop('value').to_list()).drop('value', axis=1).merge(input.groupby(input.columns.drop(dropcols).to_list()).mean(numeric_only=True), left_index=True, right_index=True).reset_index().drop_duplicates()
            #else:
                #input_nuts = input.drop(dropcols, axis=1).drop_duplicates()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld_flat.csv', index=False)
        elif filename in ['hh_size_rev','floor_resid_ssp2_rev', 'bld_shr_arch_resid']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            dropcols = ['value', 'region_nuts']
            if 'clim' in input.columns.to_list():
                dropcols = dropcols + ['clim']
            if 'urt' in input.columns.to_list():
                dropcols = dropcols + ['urt']
            #mean weighted by dwellings per NUTS, rurality and year
            input_nuts = input.set_index(input.columns.drop('value').to_list()).drop('value', axis=1).merge((input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(dropcols)), axis=0).sum(numeric_only=True), left_index=True, right_index=True).droplevel('region_nuts', axis=0).reset_index().drop_duplicates()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld_flat.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['stock_baseyear_resid_rev_share']:          
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            #mean weighted by dwellings per NUTS, rurality and year
            dw_weights_clim = dw_weights_clim.xs(2020, level='year')
            input_nuts = input.set_index(input.columns.drop('value').to_list()).drop('value', axis=1).merge((input.set_index(input.columns.drop('value').to_list()) * dw_weights_clim).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts', 'urt', 'clim'])), axis=0).sum(numeric_only=True), left_index=True, right_index=True).droplevel('region_nuts', axis=0).reset_index().drop_duplicates()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld_flat.csv', index=False)
            dw_weights_clim = dwellings.div(dwellings.groupby(level=['region_bld', 'year', 'mat']).sum())

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                                        
            print(filename + ' DONE')
        elif filename not in ['regions_R61','climatic_zones_rev', 'pop_clim_rev_SSP2', 'hh_size_rev','floor_resid_ssp2_rev', 'bld_shr_arch_resid', 'stock_baseyear_resid_rev_share']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            #mean weighted by dwellings per NUTS, rurality and year
            if filename in ['heat_intensity_rev', 'cool_intensity_rev', 'cool_days', 'shr_need_cool_resid_rev', 'shr_need_heat_resid_rev']:
                dw_weights_ = dw_weights_clim.xs(2020, level='year')
            else:
                dw_weights_ = dw_weights_clim.copy()
            
            if 'urt' in input.columns.to_list():
                dropcols = ['value', 'urt', 'clim']
            else:
                dropcols = ['value', 'clim']
            input_nuts = input.set_index(input.columns.drop('value').to_list()).drop('value', axis=1).merge((input.set_index(input.columns.drop('value').to_list()) * dw_weights_).reset_index().groupby(by=list(input.columns.drop(dropcols+['region_nuts'])), axis=0).sum(numeric_only=True), left_index=True, right_index=True).droplevel('region_nuts', axis=0).reset_index().drop_duplicates()
            input_bld = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'.csv')
            #input_bld = input_bld.groupby(input_bld.columns.drop(dropcols).to_list()).mean(numeric_only=True)
            input_bld = input_bld.set_index(input_bld.columns.drop('value').to_list())
            input_nuts = input_nuts.set_index(input_nuts.columns.drop('value').to_list())
            input_nuts.update(input_bld, overwrite=False)
            input_nuts = input_nuts.reset_index().drop_duplicates()
            if filename in ['shr_need_cool_resid_rev', 'shr_need_heat_resid_rev']:
                input_nuts.loc[input_nuts['value']>1, 'value'] = input_nuts['value']/2
                input_nuts['value'] = input_nuts['value'].round(5)
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld_flat.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['pop_clim_rev_SSP2']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts', 'urt', 'clim'])), axis=0).sum(numeric_only=True).reset_index()
            input_nuts = input_nuts.merge(input.groupby(['region_bld', 'clim']).sum(numeric_only=True).drop('year', axis=1).unstack('clim').droplevel(0, axis=1).idxmax(1).rename('clim'), left_on='region_bld', right_index=True)
            input_nuts = input_nuts.merge(input.groupby(['region_bld', 'urt']).sum(numeric_only=True).drop('year', axis=1).unstack('urt').droplevel(0, axis=1).idxmax(1).rename('urt'), left_on='region_bld', right_index=True)
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld_flat.csv', index=False)

            input_nuts.drop(['year', 'value'], axis=1).drop_duplicates().to_csv('data/input_csv_NUTS_2025_resid/climatic_zones_rev'+resolution+'_bld_flat.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['regions_R61']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            input_nuts = input.drop('region_nuts', axis=1).drop_duplicates()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld_flat.csv', index=False)

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_52991/3653874821.py:20: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  dwellings = pop.groupby(by=pop.columns.drop(['clim','value']).to_list()).sum(numeric_only=True).mul(1e6).div((hhsize*arch_shr).groupby(level=[0,1,2,4,5]).sum()).fillna(method='bfill')
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_52991/3653874821.py:118: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts', 'urt', 'clim'])), axis=0).sum(numeric_only=True).reset_index()


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
pop_clim_rev_SSP2 DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_52991/3653874821.py:51: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = input.set_index(input.columns.drop('value').to_list()).drop('value', axis=1).merge((input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(dropcols)), axis=0).sum(numeric_only=True), left_index=True, right_index=True).droplevel('region_nuts', axis=0).reset_index().drop_duplicates()


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
hh_size_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_52991/3653874821.py:51: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = input.set_index(input.columns.drop('value').to_list()).drop('value', axis=1).merge((input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(dropcols)), axis=0).sum(numeric_only=True), left_index=True, right_index=True).droplevel('region_nuts', axis=0).reset_index().drop_duplicates()


...

Missing values:  0
Null before:  1040
Null after:  88
Duplicates:  0
shr_need_cool_resid_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_52991/3653874821.py:94: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = input.set_index(input.columns.drop('value').to_list()).drop('value', axis=1).merge((input.set_index(input.columns.drop('value').to_list()) * dw_weights_).reset_index().groupby(by=list(input.columns.drop(dropcols+['region_nuts'])), axis=0).sum(numeric_only=True), left_index=True, right_index=True).droplevel('region_nuts', axis=0).reset_index().drop_duplicates()


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
shr_need_heat_resid_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_52991/3653874821.py:68: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = input.set_index(input.columns.drop('value').to_list()).drop('value', axis=1).merge((input.set_index(input.columns.drop('value').to_list()) * dw_weights_clim).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts', 'urt', 'clim'])), axis=0).sum(numeric_only=True), left_index=True, right_index=True).droplevel('region_nuts', axis=0).reset_index().drop_duplicates()


Missing values:  0
Null before:  2202
Null after:  0
Duplicates:  0
stock_baseyear_resid_rev_share DONE


In [ ]:
    filename = 'bld_shr_arch_resid'
    arch = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
    arch.drop('mat', axis=1, inplace=True)
    arch = arch.set_index(arch.columns.drop('value').to_list())

    filename = 'hh_size_rev'
    input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
    input = input.set_index(input.columns.drop('value').to_list())

    input_agg = (arch*input).groupby(level=[0,1,2,4]).sum().reset_index()

    input_agg.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv', index=False)

    for filename in input_list:
        if filename in ['hh_size_rev']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            #mean weighted by dwellings per NUTS, rurality and year
            input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)
            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')

Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
hh_size_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_34296/2335005243.py:19: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()


In [ ]:
    #Aggregating NUTS3 files back to NUTS0='region_bld'
    input_list = pd.read_csv('data/input_list_resid_2025_11_06_nuts.csv')['SSP2-NUTS'].dropna().to_list()
    input_list = [i.removesuffix('_nuts') for i in input_list]

    #calculating dwelling weights of dwellings in NUTS per country
    pop = pd.read_csv('data/input_csv_NUTS_2025_resid/pop_clim_rev_SSP2'+resolution+'.csv')
    pop['mat'] = 'perm'
    hhsize = pd.read_csv('data/input_csv_NUTS_2025_resid/hh_size_rev'+resolution+'.csv')
    hhsize['mat'] = 'perm'
    hhsize = hhsize.set_index(keys=hhsize.columns.drop('value').to_list())
    arch_shr = pd.read_csv('data/input_csv_NUTS_2025_resid/bld_shr_arch_resid'+resolution+'.csv')
    arch_shr = arch_shr.set_index(keys=arch_shr.columns.drop('value').to_list())

    dwellings = pop.groupby(by=pop.columns.drop(['clim','value']).to_list()).sum(numeric_only=True).mul(1e6).div((hhsize*arch_shr).groupby(level=[0,1,2,4,5]).sum()).fillna(method='bfill')
    dwellings = pd.merge(dwellings.reset_index(), region_nuts_clim)
    dwellings = dwellings.set_index(dwellings.columns.drop('value').to_list())

    dw_weights = dwellings.div(dwellings.groupby(level=['region_bld', 'urt', 'year', 'mat']).sum()).droplevel('clim')
    dw_weights_clim = dwellings.div(dwellings.groupby(level=['region_bld', 'urt', 'year', 'mat', 'clim']).sum())

    for filename in input_list:
        if filename in ['hh_size_rev','floor_resid_ssp2_rev', 'bld_shr_arch_resid']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            #mean weighted by dwellings per NUTS, rurality and year
            input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['stock_baseyear_resid_rev_share']:          
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            #mean weighted by dwellings per NUTS, rurality and year
            dw_weights_clim = dw_weights_clim.xs(2020, level='year')
            input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights_clim).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)
            dw_weights_clim = dwellings.div(dwellings.groupby(level=['region_bld', 'urt', 'year', 'mat', 'clim']).sum())

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                                        
            print(filename + ' DONE')
        elif filename not in ['regions_R61','climatic_zones_rev', 'pop_clim_rev_SSP2', 'hh_size_rev','floor_resid_ssp2_rev', 'bld_shr_arch_resid', 'stock_baseyear_resid_rev_share']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            #mean weighted by dwellings per NUTS, rurality and year
            if filename in ['heat_intensity_rev', 'cool_intensity_rev', 'cool_days', 'shr_need_cool_resid_rev', 'shr_need_heat_resid_rev']:
                dw_weights_ = dw_weights_clim.xs(2020, level='year')
            else:
                dw_weights_ = dw_weights_clim.copy()
            input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights_).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True)
            input_bld = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'.csv')
            input_bld = input_bld.set_index(input_bld.columns.drop('value').to_list())
            input_nuts.update(input_bld, overwrite=False)
            input_nuts = input_nuts.reset_index()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['pop_clim_rev_SSP2']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['regions_R61','climatic_zones_rev']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'.csv')
            
            input_nuts = input.drop('region_nuts', axis=1).drop_duplicates()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+resolution+'_bld.csv', index=False)
                            
            print(filename + ' DONE')

no values, only index:
    climatic_zones_rev Index(['region_bld', 'urt', 'clim'], dtype='object')
    ct_bld Index(['region_gea', 'urt', 'arch', 'mat'], dtype='object') !!!
    ct_inc_cl Index(['urt', 'inc_cl'], dtype='object') !!!

no difference:
    discount_rate_new Index(['region_gea', 'urt', 'arch', 'value'], dtype='object') !!!
    discount_rate_ren Index(['region_gea', 'urt', 'arch', 'tenr', 'value'], dtype='object') !!!

only tiny differences:
    hh_size_rev Index(['region_bld', 'urt', 'year', 'value'], dtype='object') #only tiny differences in Croatia and non-EU regions
    cool_days Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'value'], dtype='object') #not in Europe

relevant differences:
    averages/rates/shares (to be aggregated with weighted mean):
        floor_resid_ssp2_rev Index(['region_bld', 'urt', 'arch', 'mat', 'year', 'value'], dtype='object')
        shr_hh_tenr Index(['region_gea', 'urt', 'mat', 'tenr', 'year', 'value'], dtype='object')   !!!
        bld_shr_arch_resid Index(['region_gea', 'urt', 'mat', 'arch', 'year', 'value'], dtype='object')
        bld_share_mat_resid_ssp2_rev Index(['region_bld', 'clim', 'urt', 'inc_cl', 'mat', 'year', 'value'], dtype='object') #maybe only for non-EU
        bld_shr_access_cool_resid_ssp2_rev Index(['region_bld', 'clim', 'urt', 'inc_cl', 'year', 'value'], dtype='object')
        bld_shr_district_heat_resid Index(['region_gea', 'urt', 'value'], dtype='object') !!!
        bld_share_fuel_heat_resid Index(['region_gea', 'urt', 'fuel_heat', 'value'], dtype='object') !!!
        material_int_resid Index(['region_gea', 'urt', 'arch', 'material', 'year', 'value'], dtype='object') !!!
        heat_intensity_rev Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'value'], dtype='object') # I do not understand why?
        cool_intensity_rev Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'value'], dtype='object')
        shr_need_cool_resid_rev Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'value'], dtype='object') #only in few EU countries
        shr_need_heat_resid_rev Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'value'], dtype='object') #only in few EU countries
    absolute values (to be aggregated by summation):
        stock_baseyear_resid_rev_share Index(['region_bld', 'region_gea', 'urt', 'clim', 'mat', 'arch', 'yr_con',
            'bld_age', 'value'],
            dtype='object')
        pop_clim_rev_SSP2 Index(['region_bld', 'urt', 'clim', 'year', 'value'], dtype='object')


